In [1]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(
    "facebook/dinov2-small"
)

print("Model type:", config.model_type)
print("Hidden size:", config.hidden_size)
print("Patch size:", config.patch_size)
print("Layers:", config.num_hidden_layers)
print("Attention heads:", config.num_attention_heads)
print("Image size:", config.image_size)

g:\NeuroMed-VLM\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
g:\NeuroMed-VLM\.conda\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\49851\.cache\huggingface\hub\models--facebook--dinov2-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this artic

Model type: dinov2
Hidden size: 384
Patch size: 14
Layers: 12
Attention heads: 6
Image size: 518


In [2]:
import torch
from transformers import AutoModel

model = AutoModel.from_pretrained("facebook/dinov2-small")

print(type(model))
print("Total parameters:", sum(p.numel() for p in model.parameters()))

g:\NeuroMed-VLM\.conda\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\49851\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 223/223 [00:00<00:00, 15970.80it/s]

<class 'transformers.models.dinov2.modeling_dinov2.Dinov2Model'>
Total parameters: 22056576


In [3]:
model.eval()

Dinov2Model(
  (embeddings): Dinov2Embeddings(
    (patch_embeddings): Dinov2PatchEmbeddings(
      (projection): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Dinov2Encoder(
    (layer): ModuleList(
      (0-11): 12 x Dinov2Layer(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attention): Dinov2Attention(
          (q_proj): Linear(in_features=384, out_features=384, bias=True)
          (k_proj): Linear(in_features=384, out_features=384, bias=True)
          (v_proj): Linear(in_features=384, out_features=384, bias=True)
          (o_proj): Linear(in_features=384, out_features=384, bias=True)
        )
        (layer_scale1): Dinov2LayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): Dinov2MLP(
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=384, out_features=1536, bias=

In [6]:
from datasets import load_dataset
from itertools import islice
dataset = load_dataset(
    "eltorio/ROCOv2-radiology",
    streaming = True,
    split = "train"
)

dataset_shuffle = dataset.shuffle(seed = 42, buffer_size = 5000)
samples = list(islice(dataset_shuffle,1000))

In [8]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
image = samples[0]["image"].convert("RGB")

pixel_values = transform(image)

print(pixel_values.shape)
pixel_values = pixel_values.unsqueeze(dim=0)

print(pixel_values.shape)

torch.Size([3, 224, 224])
torch.Size([1, 3, 224, 224])


In [9]:
with torch.no_grad():
    outputs = model(pixel_values=pixel_values)

print("Last hidden states:", outputs.last_hidden_state.shape)

Last hidden states: torch.Size([1, 257, 384])


In [10]:
hidden = outputs.last_hidden_state

cls_embedding = hidden[:, 0, :]

patch_embeddings = hidden[:, 1:, :]

print("CLS:", cls_embedding.shape)
print("Patches:", patch_embeddings.shape)

CLS: torch.Size([1, 384])
Patches: torch.Size([1, 256, 384])


In [11]:
from transformers import AutoConfig

text_model_name = (
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
)

text_config = AutoConfig.from_pretrained(text_model_name)

print("Model type:", text_config.model_type)
print("Hidden size:", text_config.hidden_size)
print("Layers:", text_config.num_hidden_layers)
print("Attention heads:", text_config.num_attention_heads)
print("Vocabulary size:", text_config.vocab_size)
print("Max positions:", text_config.max_position_embeddings)

Model type: bert
Hidden size: 768
Layers: 12
Attention heads: 12
Vocabulary size: 30522
Max positions: 512


In [12]:
from transformers import AutoModel

text_model = AutoModel.from_pretrained(text_model_name)
text_model.eval()

print("Total parameters:", sum(p.numel() for p in text_model.parameters()))

g:\NeuroMed-VLM\.conda\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\49851\.cache\huggingface\hub\models--microsoft--BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16576.30it/s]
[transformers]

Total parameters: 109482240


In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(text_model_name)

caption = "Axial brain MRI showing an intracranial lesion."

encoded = tokenizer(
    caption,
    return_tensors="pt"
)

with torch.no_grad():
    text_outputs = text_model(**encoded)

print("Input IDs:", encoded["input_ids"].shape)
print("Hidden states:", text_outputs.last_hidden_state.shape)

Input IDs: torch.Size([1, 10])
Hidden states: torch.Size([1, 10, 768])


In [15]:
hidden = text_outputs.last_hidden_state

text_cls = hidden[:, 0, :]
mask = encoded["attention_mask"]
mask_expanded = mask.unsqueeze(-1)

print("CLS shape:", text_cls.shape)
print("Expanded mask:", mask_expanded.shape)

CLS shape: torch.Size([1, 768])
Expanded mask: torch.Size([1, 10, 1])


In [16]:
# Step 1: Mask掉Padding位置
masked_hidden = hidden * mask_expanded

# Step 2: 沿Token维度求和
sum_hidden = masked_hidden.sum(dim=1)

# Step 3: 计算每条Caption的有效Token数量
token_counts = mask_expanded.sum(dim=1)

# Step 4: 求平均
text_mean = sum_hidden / token_counts

print("Sum shape:", sum_hidden.shape)
print("Counts shape:", token_counts.shape)
print("Mean shape:", text_mean.shape)

Sum shape: torch.Size([1, 768])
Counts shape: torch.Size([1, 1])
Mean shape: torch.Size([1, 768])


In [17]:
import torch.nn as nn

image_projection = nn.Linear(384, 256)
text_projection = nn.Linear(768, 256)

image_features = cls_embedding
text_features = text_mean

image_projected = image_projection(image_features)
text_projected = text_projection(text_features)

print("Image:", image_projected.shape)
print("Text:", text_projected.shape)

Image: torch.Size([1, 256])
Text: torch.Size([1, 256])


In [18]:
import torch.nn.functional as F
image_norm = F.normalize(image_projected, p=2, dim=-1)
text_norm = F.normalize(text_projected, p=2, dim=-1)

print("Image norm:", image_norm.norm(dim=1))
print("Text norm:", text_norm.norm(dim=1))
similarity = image_norm @ text_norm.T

print("Similarity shape:", similarity.shape)
print("Similarity:", similarity.item())

Image norm: tensor([1.], grad_fn=<LinalgVectorNormBackward0>)
Text norm: tensor([1.], grad_fn=<LinalgVectorNormBackward0>)
Similarity shape: torch.Size([1, 1])
Similarity: -0.0568389855325222
